# November 19 — Sentiment Classification
### Guided Colab lab | Observation lesson

> **JOUR 701 | Research Methods in the Age of AI**  
> **Classroom teaching dataset:** The 180 rows in this notebook were **synthetically generated** to resemble the structure of the earlier Block 3 example. The written responses are invented teaching examples; they are **not actual participants’ answers** and must not be reported as real empirical findings.

**How to run:** In Colab, use **Runtime → Run all**, or click the play button beside each code cell from top to bottom. Read the text instructions and answer the reflection prompts.

## Our research question
**Do open-ended responses differ in sentiment between the Control and Personalized message conditions?**

We will first define sentiment, make **human judgments**, use a **rule-based lexicon**, and then try a **pretrained Hugging Face transformer classifier**. A model output is a measurement that needs validation, not a ground-truth fact.

### Step 1 — Load the course dataset directly from GitHub
**No CSV download or manual upload is necessary.** The dataset is stored in the course GitHub repository's `data/` folder. Run the cell below to load the synthetic practice records automatically. The CSV must first be committed to GitHub, and Colab needs an internet connection.

**Research reminder:** Each row represents a fictional participant. The `open_response` column contains the invented open-ended response.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Load the Week 14 synthetic dataset directly from our course GitHub repository.
url = "https://raw.githubusercontent.com/KayChansiri/Course_research-methods-in-the-age-of-ai/main/data/week14_synthetic_message_responses.csv"
df = pd.read_csv(url)

print("Rows and columns:", df.shape)
print("Number of participants:", len(df))
display(df.head(5))


### Step 2 — Human coding: write down your labels BEFORE using a model
Choose **positive**, **neutral/mixed**, or **negative**. Read these examples with a partner. Do you agree? Why?

In [ ]:
examples = df.loc[[0,4,9,90,94,100], ["participant_id","condition","open_response"]].copy()
display(examples)

**Record your group’s labels (do not run a model yet):**

| Participant ID | Your label | Reason |
| --- | --- | --- |
| 1 | | |
| 5 | | |
| 10 | | |
| 91 | | |
| 95 | | |
| 101 | | |

**Talk:** What should we do with a response that is both positive and uncomfortable?

### Step 3 — Method A: lexicon-based sentiment (VADER)
A **lexicon** is a dictionary of words that have sentiment weights. VADER also uses rules for punctuation, capitalization, and some word combinations. It outputs a **compound score between −1 and +1**. This is not an LLM.

Install the package once, then run the next cell.

In [ ]:
%pip -q install vaderSentiment

In [ ]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
vader = SentimentIntensityAnalyzer()
print(vader.polarity_scores("I liked the post, but the targeting felt creepy."))

### Step 4 — Turn the VADER score into labels
We must choose a **rule** (an operational definition). The common VADER cutoffs are ≥ 0.05 for positive, ≤ −0.05 for negative, and between them for neutral. For this exercise, “neutral” may also contain mixed messages.

In [ ]:
def vader_label(text):
    score = vader.polarity_scores(str(text))["compound"]
    label = "positive" if score >= 0.05 else "negative" if score <= -0.05 else "neutral/mixed"
    return pd.Series({"vader_score": score, "vader_label": label})
df[["vader_score", "vader_label"]] = df["open_response"].apply(vader_label)
display(df[["open_response", "vader_score", "vader_label"]].head(8))

### Step 5 — Descriptive comparison between the two message conditions
A chart describes the **sample**. It does not establish causality or confirm the classifier is correct.

In [ ]:
counts = pd.crosstab(df["condition"], df["vader_label"])
percent = pd.crosstab(df["condition"], df["vader_label"], normalize="index").mul(100).round(1)
display(counts); display(percent)
percent.plot(kind="bar", stacked=True, title="VADER categories by condition (synthetic teaching data)", ylabel="Percent", rot=0)
plt.legend(title="Model label", bbox_to_anchor=(1.05,1), loc="upper left")
plt.tight_layout();plt.show()

### Step 6 — Connect with Block 3: chi-square
`condition` = categorical; `vader_label` = categorical → chi-square is a candidate test. **But** the labels are estimates, and the data are simulated. We should validate labels and assess expected cell counts before making any substantive inference.

In [ ]:
from scipy.stats import chi2_contingency
chi2, p, dof, expected = chi2_contingency(counts)
print("Minimum expected count:", round(expected.min(), 1))
print(f"Illustrative chi-square: chi2={chi2:.2f}, df={dof}, p={p:.4f}")
print("Do NOT report this as evidence from real participants.")

### Step 7 — Method B: pretrained transformer sentiment classifier (Hugging Face)
**This is a discriminative text classifier, NOT a chat-style generative LLM.** We do not write prompts to it. The model reads a response and predicts positive, neutral, or negative based on its prior training.

**Technical note:** The first run needs internet access to download model files and can take longer. We classify **18 responses**, not all 180, to make the demonstration manageable. If downloading fails, continue with VADER and the human-label activity.

In [ ]:
%pip -q install "transformers>=4.40,<6" torch

In [ ]:
from transformers import pipeline
model_id = "cardiffnlp/twitter-roberta-base-sentiment-latest"
try:
    classifier = pipeline("text-classification", model=model_id, tokenizer=model_id, device=-1)
    mini = df.iloc[:9].copy()
    mini = pd.concat([mini, df.iloc[90:99]]).copy()
    predictions = classifier(mini["open_response"].tolist(), truncation=True, max_length=128)
    mini["hf_label"] = [p["label"].lower() for p in predictions]
    mini["hf_confidence"] = [round(p["score"],3) for p in predictions]
    display(mini[["condition","open_response","vader_label","hf_label","hf_confidence"]])
except Exception as e:
    print("Hugging Face demo unavailable:", str(e)[:300])
    print("Continue with your human and VADER comparison. Do not interpret this as a model result.")

### Step 8 — Compare humans and models
Pick **two disagreements** between your group and a model. Read the original sentence, identify what was missed, and propose a coding rule. Common challenges: sarcasm, negation, mixed feelings, domain-specific language, and model training context.

In [ ]:
display(df.loc[[0,4,9,90,94,100], ["participant_id","open_response","vader_label","vader_score"]])

### Step 9 — A second communication research example (transfer task)
A news organization posts: **“The mayor says the new policy will reduce traffic.”** Students write social media comments. Decide whether **sentiment toward the policy**, **sentiment toward the mayor**, or **sentiment toward the news outlet** is your outcome. Are those the same construct?

In [ ]:
practice = [
 "I support the policy, but I do not trust the mayor.",
 "The newsroom did a good job explaining this unpopular plan.",
 "Great, another brilliant idea that will make everything worse.",
 "I need more information before I decide."
]
for t in practice:
    print(t, "=>", vader_label(t)["vader_label"])

## Exit ticket (2 minutes)
1. What was the difference between a word count and sentiment classification?
2. Name one reason a sentiment model could be wrong.
3. Which Block 3 statistical test would you consider for **message condition × sentiment category**, and why?

## Optional extension: generative LLM prompting (NOT required today)
A generative LLM could also be instructed: “Classify sentiment toward the **message itself** as positive, neutral/mixed, or negative; return only one label.” But prompts, model versions, stochasticity, and label definitions must be documented and tested against human coding. **Do not assume the model’s response is automatically more valid** than a simpler approach.

### Instructor / reproducibility notes
- Hugging Face docs: https://huggingface.co/docs/course/chapter2/2 and https://huggingface.co/docs/transformers/main/main_classes/pipelines
- CardiffNLP model card: https://huggingface.co/cardiffnlp/twitter-roberta-base-sentiment-latest
- Do not interpret the model confidence as probability the label is correct.
- The sample is synthetic; the lesson illustrates workflow, not an empirical finding.
- Option: skip transformer execution during the observed class and show the VADER/human comparison if classroom internet is slow.